## Analyze RAD/PACT Violations

The following code matches the list of RAD-converted addresses we made in 1_extract_developments.ipynb to [housing code violations](https://data.cityofnewyork.us/Housing-Development/Housing-Maintenance-Code-Violations/wvxf-dwi5/about_data), which are made public by HPD for all privately managed buildings in New York City. 
<br>
<br>
This analysis merges HPD housing code violations with RAD developments by joining on the address and borough columns. Addresses come fully intact in the list of RAD developments. The housenumber and streetname columns in the HPD dataset were combined to form a full street address. 
<br>
<br>
This is a conservative approach to joining these two datasets together. The housing authority did it's own analysis and found 15,993 violations between 1/1/21 and 9/27/25 -- by merging on unique combinations of the BIN and BBL columns -- whereas the analysis below finds 14,281 during roughly the same time period.
<br>
<br>
In New York City, street addresses may sometimes be marked with two addresses, even if referring to the same exact location. NYCHA's analysis likely accounts for those extra violations attached to rarely used address assignments. 

In [1]:
## import libraries
import pandas as pd
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from datetime import datetime, timedelta

In [2]:
## read in csv that includes developments
rad_buildings = pd.read_csv('../input/coded_files/addresses_updated_121525.csv', dtype={'bbl': 'object',
                                                                                        'bldg': 'object',
                                                                                        'zip code': 'object',
                                                                                        'cd': 'object',
                                                                                        'fc': 'object',
                                                                                        'ss': 'object',
                                                                                        'sa': 'object',
                                                                                        'cc': 'object',
                                                                                        'bin': 'object'})

In [3]:
rad_buildings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   bbl          1508 non-null   object
 1   bldg         1431 non-null   object
 2   m            1127 non-null   object
 3   og_address   1530 non-null   object
 4   zip code     1527 non-null   object
 5   cd           1527 non-null   object
 6   fc           1527 non-null   object
 7   ss           1527 non-null   object
 8   sa           1527 non-null   object
 9   cc           1527 non-null   object
 10  bin          1508 non-null   object
 11  dev_name     1531 non-null   object
 12  boro         1531 non-null   object
 13  units        1531 non-null   object
 14  transfer     1531 non-null   object
 15  new_address  1530 non-null   object
dtypes: object(16)
memory usage: 191.5+ KB


In [4]:
print(f'unique BINs:{rad_buildings.bin.nunique()}\nunique addresses:{rad_buildings.new_address.nunique()}\nunique developments:{rad_buildings.dev_name.nunique()}')

unique BINs:714
unique addresses:1525
unique developments:103


In [5]:
## read in violations
violations = pd.read_csv('../input/Housing_Maintenance_Code_Violations_20251202.csv', dtype = {'LowHouseNumber':'object',
                                                                                               'BIN':'object',
                                                                                               'HighHouseNumber':'object',
                                                                                               'HouseNumber':'object',
                                                                                               'ViolationID':'object',
                                                                                               'BuildingID':'object',
                                                                                               'BoroID':'object'})

In [6]:
violations.shape

(4143079, 41)

## Make a few changes

In [7]:
## make the column names lowercase
violations.columns = violations.columns.str.lower()

In [8]:
## combine house number with street name so that we can match with PACT development csv
violations['address'] = violations['housenumber'] + ' ' + violations['streetname']

In [9]:
## get rid of columns we don't want/need
filtered_df = violations.loc[:,['violationid', 'buildingid', 'registrationid', 'boroid', 'borough',
                                'bin','bbl','housenumber', 'lowhousenumber', 'highhousenumber', 'streetname','class',
                                'inspectiondate', 'approveddate', 'originalcertifybydate',
                                'originalcorrectbydate', 'newcertifybydate', 'newcorrectbydate',
                                'certifieddate', 'ordernumber', 'novid', 'novdescription','novissueddate',
                                'currentstatusid', 'currentstatus','currentstatusdate', 'novtype', 
                                'violationstatus', 'rentimpairing','address']]

In [10]:
## changing boro names to lowercase so we can match
boro_corrections = {'BROOKLYN':'brooklyn',
                    'MANHATTAN':'manhattan',
                    'QUEENS':'queens',
                    'BRONX':'bronx',
                    'STATEN ISLAND':'staten island'}

In [11]:
## apply the corrections
filtered_df['borough'] = filtered_df['borough'].replace(boro_corrections)

In [12]:
## renaming boro column in rad df
rad_buildings = rad_buildings.rename(columns = {'boro':'borough',
                                                'new_address':'address'})

## Merge

- left: use only keys from left frame, similar to a SQL left outer join; preserve key order.
- right: use only keys from right frame, similar to a SQL right outer join; preserve key order.
- outer: use union of keys from both frames, similar to a SQL full outer join; sort keys lexicographically.
- inner: use intersection of keys from both frames, similar to a SQL inner join; preserve the order of the left keys.
- cross: creates the cartesian product from both frames, preserves the order of the left keys.

In [13]:
## stripping of whitespace just in case, so that i can merge
rad_buildings['borough'] = rad_buildings['borough'].str.strip()
filtered_df['borough'] = filtered_df['borough'].str.strip()

In [14]:
## merging the two dfs on address bc it's the most unique measure we have for the buildings
## BBLs and BINs, for example, can be the same for different addresses
combined_df = pd.merge(filtered_df,
                       rad_buildings,
                       on = ['address','borough'],
                       how = 'left',
                       indicator=True)

In [15]:
## using the indicator column to filter for addresses that exist in both dfs
both_df = combined_df[combined_df['_merge'] == 'both']

In [16]:
both_df.shape

(14420, 45)

## Clean

In [17]:
## it looks like some addresses carried over even though there's no data for them in the actual HPD violations dataset, 
# so filtering those out based on the violationid.. which i assume all valid complaints should have...
violations_df = both_df.dropna(subset = 'violationid').reset_index(drop = True)

In [18]:
violations_df.shape

(14420, 45)

In [19]:
## change NOV issue date to datetime dtype
violations_df['novissueddate'] = violations_df['novissueddate'].astype('datetime64[ns]')
violations_df['inspectiondate'] = violations_df['inspectiondate'].astype('datetime64[ns]')
violations_df['approveddate'] = violations_df['approveddate'].astype('datetime64[ns]')
violations['originalcertifybydate'] = violations['originalcertifybydate'].astype('datetime64[ns]')
violations['originalcorrectbydate'] = violations['originalcorrectbydate'].astype('datetime64[ns]')

In [20]:
## create year columns for inspections and notifications
violations_df['inspectiondate_year'] = violations_df['inspectiondate'].dt.year
violations_df['novissued_year'] = violations_df['novissueddate'].dt.year

In [21]:
## inspect duplicated violations
unique_violations = violations_df[violations_df.duplicated(subset=['violationid'], keep=False)]
unique_violations.to_csv('../output/unique_violations_test.csv')

In [22]:
## it looks like some are duplicates, while some have the same violation id but are for violations against different HPD codes (novid)
violations_df = violations_df.drop_duplicates(subset = ['violationid','novid'], keep = "first").reset_index(drop = True)

In [23]:
violations_df.shape

(14281, 47)

In [24]:
## write to a csv file
violations_df.to_csv('../output/violations_w_developments.csv', index = False)

## Violations between January 1, 2021 and September 25, 2025

In [25]:
## change inspection date to datetime dtype
violations_df['inspectiondate'] = violations_df['inspectiondate'].astype('datetime64[ns]')

In [26]:
## use boolean indexing to select rows where the dates are between 1/1/21 and 9/25/25
five_year_df = violations_df[(violations_df['inspectiondate'] >= '2021-01-01') & (violations_df['inspectiondate'] <= '2025-09-25')].reset_index()

In [27]:
## take a peak
five_year_df.head()

,index,violationid,buildingid,registrationid,boroid,borough,bin_x,bbl_x,housenumber,lowhousenumber,...,ss,sa,cc,bin_y,dev_name,units,transfer,_merge,inspectiondate_year,novissued_year
0,0,13976091,637117,432086,4,queens,4436453,4.160020e+09,56-10,56-10,...,10,31,31,4436453,OCEAN BAY APARTMENTS (BAYSIDE),"1,395",12/31/2016,both,2021,2021.0
1,1,13978917,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,10,31,31,4436434,OCEAN BAY APARTMENTS (BAYSIDE),"1,395",12/31/2016,both,2021,2021.0
2,2,13978918,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,10,31,31,4436434,OCEAN BAY APARTMENTS (BAYSIDE),"1,395",12/31/2016,both,2021,2021.0
3,3,13978930,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,10,31,31,4436434,OCEAN BAY APARTMENTS (BAYSIDE),"1,395",12/31/2016,both,2021,2021.0
4,4,13978909,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,10,31,31,4436434,OCEAN BAY APARTMENTS (BAYSIDE),"1,395",12/31/2016,both,2021,2021.0


In [28]:
## number of violations
five_year_df.shape

(14281, 48)

In [29]:
## number of unique addresses found in this new set of violations
five_year_df.address.nunique()

564

In [30]:
## number of developments found in this new list of violations
five_year_df.dev_name.nunique()

85

In [31]:
## save it to a csv
five_year_df.to_csv('../output/112021_09252025_violations.csv')

In [32]:
## how many units across these 85 developments? 
devs_and_units = five_year_df.groupby(['dev_name','units']).size().reset_index()
devs_and_units.to_csv('../output/devs_and_units.csv')

## Pre-2021 

In [33]:
pre_21_df = pd.read_csv('../input/Housing_Maintenance_Code_Violations_20251208.csv',dtype = {'LowHouseNumber':'object',
                                                                                               'BIN':'object',
                                                                                               'HighHouseNumber':'object',
                                                                                               'HouseNumber':'object',
                                                                                               'ViolationID':'object',
                                                                                               'BuildingID':'object',
                                                                                               'BoroID':'object'})

/var/folders/t4/yhv9cps51w7bfdt0w1lkv1zm0000gn/T/ipykernel_71076/655140175.py:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  pre_21_df = pd.read_csv('../input/Housing_Maintenance_Code_Violations_20251208.csv',dtype = {'LowHouseNumber':'object',


In [34]:
## make the column names lowercase
pre_21_df.columns = pre_21_df.columns.str.lower()

In [35]:
## combine house number with street name so that we can match with PACT development csv
pre_21_df['address'] = pre_21_df['housenumber'] + ' ' + pre_21_df['streetname']

In [36]:
## apply the corrections
pre_21_df['borough'] = pre_21_df['borough'].replace(boro_corrections)

In [37]:
pre_21_combined = pd.merge(pre_21_df,
                           rad_buildings,
                           on = ['address','borough'],
                           how = 'left',
                           indicator=True)

In [38]:
both_21_df = pre_21_combined[pre_21_combined['_merge'] == 'both']

In [39]:
both_21_df.shape

(2687, 57)

In [40]:
## it looks like some are duplicates, while some have the same violation id but are for violations against different HPD codes (novid)
pre_21_dropped = both_21_df.drop_duplicates(subset = ['violationid','novid'], keep = "first").reset_index(drop = True)

In [41]:
pre_21_dropped.shape

(2660, 57)

In [42]:
## change inspection date column to datetime dtype
pre_21_dropped['inspectiondate'] = pre_21_dropped['inspectiondate'].astype('datetime64[ns]')

In [43]:
## filter for between 12/31/2016 and 1/1/21
since_ocean_bay = pre_21_dropped[(pre_21_dropped['inspectiondate'] >= '2016-12-31') & (pre_21_dropped['inspectiondate'] <= '2021-01-01')].reset_index()

In [44]:
since_ocean_bay.shape

(1120, 58)

In [45]:
since_ocean_bay.head()

,index,violationid,buildingid,registrationid,boroid,borough,housenumber,lowhousenumber,highhousenumber,streetname,...,cd,fc,ss,sa,cc,bin_y,dev_name,units,transfer,_merge
0,299,11594728,383517,309549,3,brooklyn,2058,2058,2068,UNION STREET,...,16,09,25,55,41,3080818,SUTTER AVENUE-UNION STREET,100,11/28/2023,both
1,300,11594724,383517,309549,3,brooklyn,2058,2058,2068,UNION STREET,...,16,09,25,55,41,3080818,SUTTER AVENUE-UNION STREET,100,11/28/2023,both
2,301,11598523,383517,309549,3,brooklyn,2058,2058,2068,UNION STREET,...,16,09,25,55,41,3080818,SUTTER AVENUE-UNION STREET,100,11/28/2023,both
3,302,11598501,383517,309549,3,brooklyn,2058,2058,2068,UNION STREET,...,16,09,25,55,41,3080818,SUTTER AVENUE-UNION STREET,100,11/28/2023,both
4,303,11598519,383517,309549,3,brooklyn,2058,2058,2068,UNION STREET,...,16,09,25,55,41,3080818,SUTTER AVENUE-UNION STREET,100,11/28/2023,both


## In which post-transfer years do the violations occur?

In [46]:
## creating a new date column indicating the "pull date"
## violation status updated daily
five_year_df['date_pulled'] = '2025-09-25'

In [47]:
## changing dtypes
five_year_df['date_pulled'] = five_year_df['date_pulled'].astype('datetime64[ns]')
five_year_df['currentstatusdate'] = five_year_df['currentstatusdate'].astype('datetime64[ns]')
five_year_df['transfer'] = five_year_df['transfer'].astype('datetime64[ns]')
five_year_df['inspectiondate'] = five_year_df['inspectiondate'].astype('datetime64[ns]')

In [48]:
## how many days between the transfer date and the date in which the violation was inspected?
five_year_df['vio_days_post_transf'] = five_year_df['inspectiondate'] - five_year_df['transfer']

In [49]:
## create a function to assign categories based on the number of days
def assign_post_conversion_year(days):
    if days <= timedelta(days =365):
        return 'first yr'
    elif days <= timedelta(days =730):
        return 'second yr'
    elif days <= timedelta(days =1095):
        return 'third yr'
    elif days <= timedelta(days =1460):
        return 'fourth yr'
    elif days <= timedelta(days =1825):
        return 'fifth yr'
    elif days <= timedelta(days =2190):
        return 'sixth yr'
    elif days <= timedelta(days =2555):
        return 'seventh yr'
    elif days <= timedelta(days =2920):
        return 'eigth yr'
    elif days <= timedelta(days =3285):
        return 'ninth yr'
    else:
        return '10+ years'

In [50]:
## apply the function to the df
five_year_df['vio_yrs_post_transfer'] = five_year_df['vio_days_post_transf'].apply(assign_post_conversion_year)

In [51]:
five_year_df['vio_yrs_post_transfer'].value_counts()

vio_yrs_post_transfer
first yr      3849
second yr     3240
third yr      2024
fourth yr     1446
10+ years     1003
sixth yr       883
fifth yr       869
seventh yr     650
eigth yr       219
ninth yr        98
Name: count, dtype: int64

## How long do violations stay open?

The methodology: if a violation has already been closed, then use the current status date of that violation to determine the number of days it was open. But if the violation is still open, then use our "pull date" to determine how long it's been open so far. 

In [52]:
## create a column for the number of days violations stay open
five_year_df['vio_duration_days'] = np.where(
    five_year_df['violationstatus'] == 'Close', # <-- if violation status is closed
    (five_year_df['currentstatusdate'] - five_year_df['novissueddate']), # <-- then subtract the notice of vio date from the current status date
    (five_year_df['date_pulled'] - five_year_df['novissueddate']) # <-- else subtract notice of vio date from the date data was pulled
)

In [53]:
## same as before, create a function that assigns categories based on number of days open
def assign_num_years_open(days):
    if days <= timedelta(days =7):
        return 'a week or less'
    elif days <= timedelta(days=30):
        return 'one week to a month'
    elif days <= timedelta(days =90):
        return 'one to three months'
    elif days <= timedelta(days =180):
        return 'three to six months'
    elif days <= timedelta(days =364):
        return 'six months to a year'
    else:
        return 'a year or longer'

In [54]:
## create categories/ranges for the number of days violations stay open
five_year_df['vio_open_range'] = five_year_df['vio_duration_days'].apply(assign_num_years_open)

In [55]:
## how long do ALL violations stay open?
five_year_df['vio_open_range'].value_counts().reset_index()

,vio_open_range,count
0,three to six months,3928
1,a year or longer,3691
2,one to three months,2805
3,six months to a year,2520
4,one week to a month,1088
5,a week or less,249


In [56]:
## what percentage of all violations we found are open for a year or longer?
3691/14281

0.2584552902457811

In [57]:
## how long do violations remain open at each development?
violation_length = five_year_df.groupby(['dev_name','vio_open_range']).size().reset_index(name='open_range_vios')

In [58]:
## filter for only more than a year
duration_df = violation_length[violation_length['vio_open_range']=='a year or longer']

In [59]:
duration_df.dev_name.nunique()

70

In [60]:
## make a df that has the total number of violations
tot_violations = five_year_df.dev_name.value_counts().reset_index(name='tot_vios')

In [61]:
## merge
duration_w_tot = pd.merge(duration_df,
                          tot_violations,
                          on='dev_name',
                          how='left')

In [62]:
## make a new column with the total open for more than a year as a percent of total violations
duration_w_tot['pct_open_range'] = duration_w_tot['open_range_vios']/duration_w_tot['tot_vios']

In [63]:
year_or_longer_df = duration_w_tot.sort_values(by='pct_open_range',ascending=False).reset_index()

In [64]:
year_or_longer_df.head()

,index,dev_name,vio_open_range,open_range_vios,tot_vios,pct_open_range
0,40,MARSHALL PLAZA,a year or longer,104,137,0.759124
1,22,FENIMORE-LEFFERTS,a year or longer,14,19,0.736842
2,23,FIORENTINO PLAZA,a year or longer,281,411,0.683698
3,45,"PARK AVENUE-EAST 122ND, 123RD STREETS",a year or longer,39,62,0.629032
4,49,RUTLAND TOWERS,a year or longer,25,42,0.595238


In [65]:
year_or_longer_df.to_csv('../output/year_or_longer.csv')

In [66]:
duration_w_tot.dev_name.nunique()

70

## C-Class Violations

In [67]:
## how long do Class-C violations, in particular, stay open?
violations_class_five_year = five_year_df[five_year_df['class'] == 'C']

In [68]:
violations_class_five_year.dev_name.nunique()

83

In [69]:
violations_class_five_year.address.nunique()

414

In [70]:
## what percentage of addresses have had at least one c-class violation?
414/564

0.7340425531914894

In [71]:
## save to a csv
five_year_df.to_csv('../output/violations_21_25.csv')

## How are Ocean Bay, Linden, Boulevard doing?

In [72]:
ocean_bay = five_year_df[five_year_df['dev_name'] == 'OCEAN BAY APARTMENTS (BAYSIDE)']

In [73]:
ocean_bay

,index,violationid,buildingid,registrationid,boroid,borough,bin_x,bbl_x,housenumber,lowhousenumber,...,units,transfer,_merge,inspectiondate_year,novissued_year,date_pulled,vio_days_post_transf,vio_yrs_post_transfer,vio_duration_days,vio_open_range
0,0,13976091,637117,432086,4,queens,4436453,4.160020e+09,56-10,56-10,...,"1,395",2016-12-31,both,2021,2021.0,2025-09-25,1468 days,fifth yr,77 days,one to three months
1,1,13978917,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,"1,395",2016-12-31,both,2021,2021.0,2025-09-25,1469 days,fifth yr,23 days,one week to a month
2,2,13978918,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,"1,395",2016-12-31,both,2021,2021.0,2025-09-25,1469 days,fifth yr,23 days,one week to a month
3,3,13978930,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,"1,395",2016-12-31,both,2021,2021.0,2025-09-25,1469 days,fifth yr,23 days,one week to a month
4,4,13978909,811145,432092,4,queens,4436434,4.160010e+09,51-45,51-45,...,"1,395",2016-12-31,both,2021,2021.0,2025-09-25,1469 days,fifth yr,23 days,one week to a month
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14211,14211,18239563,811183,432089,4,queens,4436450,4.160020e+09,434,434,...,"1,395",2016-12-31,both,2025,2025.0,2025-09-25,3183 days,ninth yr,8 days,one week to a month
14221,14221,18246827,811149,432097,4,queens,4436444,4.160020e+09,54-41,54-41,...,"1,395",2016-12-31,both,2025,2025.0,2025-09-25,3187 days,ninth yr,35 days,one to three months
14242,14242,18247193,955148,432085,4,queens,4436454,4.160020e+09,56-16,56-16,...,"1,395",2016-12-31,both,2025,2025.0,2025-09-25,3188 days,ninth yr,1 days,a week or less
14243,14243,18247194,955148,432085,4,queens,4436454,4.160020e+09,56-16,56-16,...,"1,395",2016-12-31,both,2025,2025.0,2025-09-25,3188 days,ninth yr,1 days,a week or less


In [74]:
ocean_bay.vio_yrs_post_transfer.value_counts()

vio_yrs_post_transfer
sixth yr      325
eigth yr      219
seventh yr    203
fifth yr      188
ninth yr       97
Name: count, dtype: int64

In [75]:
linden_df = five_year_df[five_year_df['dev_name'] == 'LINDEN']

In [76]:
linden_df.vio_yrs_post_transfer.value_counts()

vio_yrs_post_transfer
third yr     358
fourth yr    290
first yr     201
second yr    171
Name: count, dtype: int64

In [77]:
boulevard_df = five_year_df[five_year_df['dev_name'] == 'BOULEVARD']

In [78]:
boulevard_df.vio_yrs_post_transfer.value_counts()

vio_yrs_post_transfer
third yr     376
fourth yr    236
second yr    213
first yr     152
Name: count, dtype: int64